# ES — Evolution Strategies (population-based black-box optimization): Qwen2.5-0.5B-Instruct (ES backend, vLLM rollout)

In [ ]:
from aligntune.core.backend_factory import create_es_trainer

def math_answer_reward(prompt=None, response=None, reference=None, **kwargs):
    """Tiny custom reward: +1 if the (numeric) reference string appears in the response."""
    if not response:
        return 0.0
    if reference and str(reference).strip() in response:
        return 1.0
    return 0.0

# use_unsloth=True: fixed two real bugs to get this working (both in
# core/peft/lora.py's apply_to_unsloth(), shared by every unsloth trainer):
# - it read config.train.gradient_checkpointing/.seed unconditionally, but
#   UnifiedESConfig has no .train section (it's .es instead) -> AttributeError.
# - it always used the flat config.model.lora_r/lora_alpha/lora_dropout
#   fallback defaults instead of checking config.model.peft.rank/alpha/dropout
#   first (apply_to_transformers already did this, apply_to_unsloth didn't) -
#   built a rank-16 adapter while ES's vLLM rollout backend was separately
#   configured for max_lora_rank=8 (from config.model.peft.rank), crashing
#   vLLM with "LoRA rank 16 is greater than max_lora_rank 8".
# Verified working end-to-end after both fixes.
trainer = create_es_trainer(
    model_name="Qwen/Qwen2.5-0.5B-Instruct",
    dataset_name="openai/gsm8k",
    subset="main",
    split="train",
    max_samples=8,
    output_dir="./out_es",
    max_seq_length=256,
    batch_size=2,               # prompts evaluated per iteration
    learning_rate=0.01,
    population_size=2,          # TINY: reward fn evaluated population_size times/iteration
    sigma=0.1,
    num_iterations=1,           # TINY: single ES iteration
    rewards=[{"type": "custom", "params": {"function": math_answer_reward}}],
    use_peft=True,
    use_unsloth=True,
    dtype="fp16",            # T4 (Turing) lacks bf16 tensor cores; force fp16 for the vLLM engine
    max_new_tokens=32,
    num_return_sequences=1,
)

# trainer.train() now runs setup_model/setup_data/setup_reward_function/setup_rollout_backend
# internally (matching the other trainers), so no manual setup_* calls are needed here.
results = trainer.train()
print("ES training completed.")
print(results)
